In [1]:
import os

# Better memory handling for variable prompt and completion lengths.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Faster loading for models containing multiple weight shards.
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "8"

import torch

torch.backends.cuda.matmul.fp32_precision = "tf32"
torch.backends.cudnn.conv.fp32_precision = "tf32"

torch.use_deterministic_algorithms(False)
torch.backends.cudnn.deterministic = False

In [2]:
import torch 
from transformers import AutoModelForCausalLM, AutoTokenizer
from dataclasses import dataclass 
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from functools import partial
from torch.optim import AdamW
import json
import re
from pathlib import Path
from tqdm import tqdm
import wandb

/workspace/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass
class DataSample:
    context: list[dict]
    label: dict
    metadata: dict = None

In [4]:
model_name = "Qwen/Qwen3.5-2B"
device = "cuda"

In [5]:
model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        use_kernels=True
    ).to(device)

tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="left"
    )

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 803.59it/s]


# Collate Batch

The collate batch function takes a list of samples generated by the dataloader. The goal of the function is to convert this list of samples into a dictionary that contains the batched tokens and the original samples. The batched tokens contains the data as converted by the tokenizer and the original samples are included for later use.
The function applies the following transformation in exact order.


1. Loop over the list of samples
   1. Take each sample in the loop and convert it into chatml tokens. This basically implies, taking a context like 
   ```json
      [
         {
            "role": "system",
            "content": "You are a helpful assistant"
         },
         {
            "role": "user",
            "content": "What is 2 + 3"
         }
      ]
   ``` 
   and turning it into the chatml format of the form

   ```text

   <|im_start|>system\nYou are a helpful assistant<|im_end|>\n<|im_start|>user\nWhat is 2 + 3<|im_end|>\n<|im_start|>assistant\n
   
   ```

   You will notice an extra `<|im_start|>assistant\n` added to the context, this is due to the fact that we set `add_generation_prompt=True` in 
   ```python
   formatted_prompt = tokenizer.apply_chat_template(sample.context, tokenize=False, add_generation_prompt=True, enable_thinking=False)
   ```
   2. Next we repeat the prompts by the group size, in GRPO we use a group size greater than 1. We then append a list of this repeated prompts and the samples that carry them to a list.
2. Now we have a list of contexts, with each item being itself a list of repeated prompts. We then tokenize this into pytorch tensors, this is where the ordinary text gets turned into integers, and because different prompts in a batch will typically have different lengths, we need to pad to the longest, we do this by setting `padding=True` . This will return input ids and an attention mask with zeros on the padding position.  The whole process is just a way of doing this.

```python
messages1 = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "What is 2 + 3"
    }
]

messages2 = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "What is the square root of 64?"
    }
]

batch = []

formatted_prompt1 = tokenizer.apply_chat_template(messages1, add_generation_prompt=True, tokenize=False)
formatted_prompt2 = tokenizer.apply_chat_template(messages2, add_generation_prompt=True, tokenize=False)
batch.append(formatted_prompt1)
batch.append(formatted_prompt2)


tokens = tokenizer(
    batch,
    return_tensors="pt",
    padding=True
)
```
And if you `print(tokens)`, you get this

```python
{
        'input_ids': tensor([
            [    2,     2,     2,     1,  9690,   198,  2683,   359,   253,  5356,
                11173,     2,   198,     1,  4093,   198,  1780,   314,   216,    34,
                1232,   216,    35,     2,   198,     1,   520,  9531,   198],
            [    1,  9690,   198,  2683,   359,   253,  5356, 11173,     2,   198,
                1,  4093,   198,  1780,   314,   260,  5222,  3752,   282,   216,
                38,    36,    47,     2,   198,     1,   520,  9531,   198]
            ]), 
        'attention_mask': tensor([
            [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
            1, 1, 1, 1, 1],

            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
            1, 1, 1, 1, 1]
            ])}
```

Pay attention to the `attention_mask`, the first prompt is shoter than the second one by 3 tokens, hence, it is padded at the start with 3 padding tokens, you can see this clearly with its attention mask having `0, 0, 0]`  at the start. If you look closely at its `input_ids` you will see it starts with `2 , 2, 2]` which are the padding tokens. If you look at the second prompt, you will see it doesnt have the same padding tokens and its attention mask is all `1s` as it is the longest prompt and needs no padding.

At the end of this simple function which we have explained rather elaborately, we return the tokens and the original list of samples from which they were contructed ( we will need them too)




In [6]:
THINKING_MODE = True

In [7]:
def collate_batches(samples: list[DataSample], tokenizer: AutoTokenizer, group_size: int = 1):

    all_contexts = [] 
    all_samples = []
    for sample in samples:
        formatted_prompt = tokenizer.apply_chat_template(sample.context, tokenize=False, 
                                                         add_generation_prompt=True, 
                                                         enable_thinking=THINKING_MODE)

        group_prompts = [formatted_prompt for _ in range(group_size)]
        group_samples = [sample for _ in range(group_size)]

        all_contexts.extend(group_prompts)
        all_samples.extend(group_samples)

    tokens = tokenizer(
        all_contexts,
        return_tensors="pt",
        padding=True
    )

    return {
        "samples": all_samples,
        "tokens": tokens
    }

# Sampler

The `generate_batch` function is where the model generates the data we use for training. 

The function does the following

1. we pick the `prompt_ids` and `attention_mask` from the batch
2. The model generates completions in batch for all the prompts, which at this point will be the `number of unique prompts` x `group_size`, recall in the previous function we repeated each prompt by `group_size` times. Note, during generation from a batch of prompts, the completions will typically vary in length, even within the same group of prompts, some completions can be 100 tokens, and others 200 tokens. Therefore, we specify a `pad_token_id`, this ensures, the full completions will be of shape `[number of prompts, lenght of the longest completion ]` . Later on, we need to carefully construct a mask to mask out our RL loss on the padded completions.

The output of the `model.generate` will include the full prompt + generated outputs, to get the generated tokens, we will slice out the input ids with `generated_outputs = outputs[:, input_ids.shape[1]:]` this works since all the prompts in our batch are padded to the same length, and the outputs are all right padded to the longest generation too.

We can then get the actual text by decoding the tokens with `tokenizer.batch_decode`, which gives us one string per completion in the batch. Note that the `generated_texts` will not contain the padding tokens, as `skip_special_tokens=True` will cause them to not be decoded to text. It is this text we will need to compute our rewards later. However, the `generated_outputs` will contain the padding tokens.

One extra flag worth paying attention to is `do_sample`. During training we set `do_sample=True`, this makes the model sample from its probability distribution, so the completions within a group will differ from each other. This diversity is the whole point of GRPO, if all 8 completions in a group were identical, they would all get the same reward and there would be nothing to learn from. During evaluation however, we set `do_sample=False`, the model then greedily picks the most likely token at every step, which makes the eval deterministic and comparable across training steps.

In [8]:
def generate_batch(batch: dict, max_new_tokens: int = 256, do_sample: bool = True):

    samples = batch["samples"]
    tokens = batch["tokens"].to(model.device)

    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        do_sample=do_sample,
        temperature=1,
        use_cache=True,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id
    )

    generated_outputs = outputs[:, input_ids.shape[1]:]

    generated_texts = tokenizer.batch_decode(
        generated_outputs,
        skip_special_tokens=True
    )

    return {
        "samples": samples,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "outputs": outputs,
        "generated_outputs": generated_outputs,
        "generated_texts": generated_texts
    }

# Dataset

The dataset is a standard pytorch `Dataset` and it is refreshingly simple. Each line of the jsonl file is one training sample, and looks like this

```json
{
   "context": [
      {"role": "system", "content": "You are a helpful assistant..."},
      {"role": "user", "content": "What is 47 + 15?"}
   ],
   "label": {"answer": "62"},
   "metadata": {}
}
```

Notice what is *not* here, there is no target completion. In SFT we would need the full assistant response to imitate, but in RL the model writes its own completions and the only ground truth we need is the final answer to score them against. This is one of the quiet superpowers of RL, the data is much cheaper to make.

The class simply reads every line, parses the json and wraps it in the `DataSample` dataclass we defined earlier. The `max_items` argument lets us truncate the dataset for quick smoke tests, `-1` means use everything.

In [9]:

class RLDataset(Dataset):

    def __init__(self, dataset_file: str, max_items: int=-1):
        
        self.dataset = []

        with open(dataset_file, mode="r") as f:
            rows = f.readlines()

        if max_items != -1:
            rows = rows[:max_items]

        for row in rows:

            data = json.loads(row)

            sample = DataSample(
                context=data["context"],
                label=data["label"],
                metadata=data["metadata"] if "metadata" in data.keys() else None
            )

            self.dataset.append(
                sample
            )


    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index: int) -> DataSample:

        item = self.dataset[index]

        return item

In [10]:

# Non-greedy so the first complete tag pair wins, DOTALL so a stray newline
# inside the tags does not hide an otherwise correct answer.
ANSWER_PATTERN = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)

def normalise_answer(text: str) -> str:
    """Fold away the differences nobody wants to punish: case, surrounding
    whitespace and a trailing full stop."""
    return " ".join(str(text).strip().lower().split()).rstrip(".")


def answers_match(predicted: str, expected: str) -> bool:

    if predicted == expected:
        return True

    # "07" and "7.0" are the right answer spelled differently, not a wrong one.
    try:
        return float(predicted) == float(expected)
    except ValueError:
        return False


def reward_func_non_thinking(sample: DataSample, generated_response: str) -> float:

    if not generated_response:
        return 0.0

    matches = ANSWER_PATTERN.findall(generated_response)

    # Exactly one tag pair, so listing several guesses cannot beat committing
    # to one of them.
    if len(matches) != 1:
        return 0.0

    reward = 0.0

    if answers_match(normalise_answer(matches[0]), normalise_answer(sample.label["answer"])):
        reward += 1.0

    return reward


def reward_func_thinking(sample: DataSample, generated_response: str) -> float:

    reward = 0.0
    if not generated_response:
        return reward

    if generated_response.count("</think>") != 1 or generated_response.count("<think>") > 1:
        return reward

    close = generated_response.index("</think>")
    open_index = generated_response.find("<think>")

    if open_index == -1:
        think_text = generated_response[:close]
    elif open_index < close:
        think_text = generated_response[open_index + len("<think>"):close]
    else:
        return reward

    if not think_text.strip():
        return reward

    body = generated_response[close + len("</think>"):]

    matches = ANSWER_PATTERN.findall(body)

    if len(matches) != 1:
        return reward


    if answers_match(normalise_answer(matches[0]), normalise_answer(sample.label["answer"])):
        reward += 1.0
        
    return reward

# Checkpointing

RL runs routinely get worse before they get better, so the final step of training is often not the best one. Rather than saving a directory per improvement, we overwrite a single best-so-far checkpoint, the point is to finish the run holding the strongest model. The tokenizer is saved alongside the weights so the checkpoint directory can be loaded on its own, and a small `checkpoint.json` records which step it came from and what eval reward it achieved.

In [11]:
def save_checkpoint(model, tokenizer, checkpoint_dir: Path, step: int, eval_reward: float) -> None:
    """Overwrite the best-so-far checkpoint.

    Only the best is kept rather than one directory per improvement: the point
    is to finish the run holding the strongest model, and RL runs routinely get
    worse before they get better, so the final step is often not the best one.
    The tokenizer goes with it so the directory can be loaded on its own.
    """
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)

    (checkpoint_dir / "checkpoint.json").write_text(
        json.dumps({"step": step, "eval_mean_reward": eval_reward}, indent=2),
        encoding="utf-8"
    )

    print(f"Saved checkpoint: step {step}, eval mean reward {eval_reward:.3f} -> {checkpoint_dir}")


In [12]:
DATA_DIR = Path("data")

eps_low = 0.2
eps_high = 0.25
group_size = 8
max_trials = 2
lr = 5e-6
ppo_epoch = 1
train_batch_size=6 if THINKING_MODE else 32
test_batch_size=64
eval_steps = 10

reward_func = reward_func_thinking if THINKING_MODE else reward_func_non_thinking

# Room for the answer tags. Too small a budget and the response is cut off
# before </answer>, which scores zero however good the reasoning was.
max_new_tokens = 192 if THINKING_MODE else 32

optimizer = AdamW(model.parameters(), lr=lr, fused=True)

training_dataset = RLDataset(dataset_file=DATA_DIR / "code_train.jsonl")

eval_dataset = RLDataset(dataset_file=DATA_DIR / "code_test.jsonl")

maths_dataset = RLDataset(dataset_file=DATA_DIR / "multiplication_test.jsonl")

dataloader = DataLoader(
    dataset=training_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    collate_fn=partial(collate_batches, tokenizer=tokenizer, group_size=group_size)
)

eval_dataloader = DataLoader(
    dataset=eval_dataset,
    batch_size=test_batch_size,
    shuffle=False,
    collate_fn=partial(collate_batches, tokenizer=tokenizer, group_size=1)
)

maths_dataloader = DataLoader(
    dataset=maths_dataset,
    batch_size=test_batch_size,
    shuffle=False,
    collate_fn=partial(collate_batches, tokenizer=tokenizer, group_size=1)
)


checkpoint_dir = Path("checkpoints") / "grpo"
best_eval_reward = float("-inf")

In [13]:
run = wandb.init(
    project="standard-grpo",
    config={
        "model_name": model_name,
        "thinking_mode": THINKING_MODE,
        "eps_low": eps_low,
        "eps_high": eps_high,
        "group_size": group_size,
        "train_batch_size": train_batch_size,
        "test_batch_size": test_batch_size,
        "lr": lr,
        "ppo_epoch": ppo_epoch,
        "max_trials": max_trials,
        "max_new_tokens": max_new_tokens,
        "eval_steps": eval_steps,
    }
)

# Two clocks: batch-level metrics tick once per generated batch, while the
# loss and grad norm tick once per optimizer step (ppo_epoch steps per batch).
wandb.define_metric("batch")
wandb.define_metric("ppo_step")
wandb.define_metric("reward/*", step_metric="batch")
wandb.define_metric("train/*", step_metric="batch")
wandb.define_metric("loss/*", step_metric="ppo_step")
wandb.define_metric("grad/*", step_metric="ppo_step")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: johnolafenwa to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
EVAL_SEED = 0

def evaluate(step: int):

    global best_eval_reward

    cpu_rng_state = torch.random.get_rng_state()
    cuda_rng_state = torch.cuda.get_rng_state()
    torch.manual_seed(EVAL_SEED)

    val_reward = []

    for batch in eval_dataloader:

        generated_batch = generate_batch(
            batch,
            max_new_tokens=max_new_tokens,
            do_sample=THINKING_MODE
        )

        rewards = [
            reward_func(sample, generated_text)
            for sample, generated_text in zip(generated_batch["samples"], generated_batch["generated_texts"])
        ]

        batch_mean_reward = sum(rewards) / len(rewards)

        val_reward.append(batch_mean_reward)

    torch.random.set_rng_state(cpu_rng_state)
    torch.cuda.set_rng_state(cuda_rng_state)

    avg_val_reward = sum(val_reward) / len(val_reward)
    print(f"Eval: mean reward: {avg_val_reward}")

    # Strictly better, so a plateau does not rewrite the checkpoint. Eval
    # reward is quantised in coarse steps here, so ties are common.
    if avg_val_reward > best_eval_reward:
        best_eval_reward = avg_val_reward

        # Step 0 is the untrained baseline. It sets the bar to beat, but
        # writing it out would only copy the pretrained weights to disk,
        # so any checkpoint that does exist has genuinely beaten it.
        if step > 0:
            save_checkpoint(model, tokenizer, checkpoint_dir, step, avg_val_reward)

    wandb.log({
        "reward/eval_mean_reward": avg_val_reward,
        "reward/best_eval_reward": best_eval_reward,
        "batch": step
    })

In [15]:

def maths_evaluate(step: int):

    cpu_rng_state = torch.random.get_rng_state()
    cuda_rng_state = torch.cuda.get_rng_state()
    torch.manual_seed(EVAL_SEED)

    val_reward = []

    for batch in maths_dataloader:

        generated_batch = generate_batch(
            batch,
            max_new_tokens=max_new_tokens,
            do_sample=THINKING_MODE
        )

        rewards = [
            reward_func(sample, generated_text)
            for sample, generated_text in zip(generated_batch["samples"], generated_batch["generated_texts"])
        ]

        batch_mean_reward = sum(rewards) / len(rewards)

        val_reward.append(batch_mean_reward)

    torch.random.set_rng_state(cpu_rng_state)
    torch.cuda.set_rng_state(cuda_rng_state)

    avg_val_reward = sum(val_reward) / len(val_reward)
    print(f"Maths Eval: mean reward: {avg_val_reward}")

    wandb.log({
        "reward/maths_eval_mean_reward": avg_val_reward,
        "batch": step
    })

In [16]:
# eval before training
evaluate(0)

Eval: mean reward: 0.03125


In [17]:
maths_evaluate(0)

Maths Eval: mean reward: 0.0


# The GRPO Training Loop

This is where everything comes together. For every batch the loop does the following, in exact order.

### 1. Generate and score

We sample a group of completions with `generate_batch` and score each one with `reward_func`. Nothing here requires gradients, the model is simply acting.

### 2. Build the masks

This is the fiddly part, and it is worth slowing down for. Remember `outputs` is `[prompt + completion]` padded on both sides, left padding on the prompt and right padding on the completion. We need two masks

- `label_masks` marks the positions the loss is allowed to touch, we only want to train on the tokens the model actually generated, not the prompt and not the padding.
- `input_attention_mask` marks the positions the model is allowed to attend to when we recompute log probabilities, prompt padding and completion padding are both switched off.

The trick for finding where each completion really ends is this pair of lines

```python
eos_tokens = generated_outputs.eq(tokenizer.eos_token_id)
first_eos_token = (eos_tokens.cumsum(dim=-1) - eos_tokens.long()) == 0
```

`eos_tokens` is `True` at EOS-token positions and `False` everywhere else. The cumsum expression is then `1` up to and *including* the first EOS token and `0` on the padding after it, whether or not the padding token happens to be the EOS token (in this tokenizer it is, which is exactly why we cannot just mask out pad tokens directly). We include the first EOS in the loss on purpose, stopping is a decision the model makes and it should be trained on it.

### 3. Compute the advantages

For each group of `group_size` completions we subtract the group mean reward from each completion's reward

```python
group_advantages = [reward - group_mean for reward in group_rewards]
```

That is the whole of the "GR" in GRPO. A completion is not judged by its absolute reward but by whether it did better or worse than its siblings sampled from the same prompt. If every completion in the group scored the same, all advantages are zero, the batch teaches nothing, and we skip it rather than pay for two useless gradient steps. We log this as `train/skipped_batch`, a long run of skipped batches early on usually means the reward is too hard to reach and the model never gets off the ground.

### 4. The PPO update

Before computing the loss we shift everything by one, `input_ids = outputs[:, :-1]` and `label_ids = outputs[:, 1:]`, the standard next-token setup, the logits at position `t` predict the token at position `t+1`.

Then for `ppo_epoch` epochs we recompute the per-token log probabilities and form the clipped surrogate objective

```python
ratio = torch.exp(new_token_log_probs - old_token_log_probs)

unclipped_loss = ratio * advantages
clipped_loss = torch.clamp(ratio, min=1-eps, max=1+eps) * advantages

ppo_objective = torch.min(unclipped_loss, clipped_loss)
```

Pay attention to the first epoch, `old_token_log_probs` is captured (detached) from the very first forward pass, so on epoch 0 the ratio is exactly 1 everywhere and the loss reduces to plain REINFORCE with a baseline. It is only on the second epoch, when the policy has moved but the data has not, that the ratio drifts from 1 and the clipping starts doing its job of keeping the update honest.

The `torch.min` is the pessimistic choice, whichever of the clipped and unclipped objectives promises less improvement is the one we optimise, so the model can never profit from pushing the ratio outside the trust region.

Finally we mask the per-token loss with `label_masks`, average over each sequence's real generated tokens (the `clamp_min(1)` guards against dividing by zero on a completely empty completion), take the mean over the batch, and do a standard clipped-gradient optimizer step. The loss is negated because optimizers minimise and we want to *maximise* the objective.

In [ ]:
global_step = 0

for idx, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
   
    for trial in range(1, max_trials + 1):

        with torch.no_grad():
            generated_batch = generate_batch(batch, max_new_tokens=max_new_tokens)
        

        samples = generated_batch["samples"]
        input_ids = generated_batch["input_ids"]
        attention_mask = generated_batch["attention_mask"]
        outputs = generated_batch["outputs"]
        generated_outputs = generated_batch["generated_outputs"]
        generated_texts = generated_batch["generated_texts"]

        # Score every completion against the sample it was generated for
        rewards = [
            reward_func(sample, generated_text)
            for sample, generated_text in zip(samples, generated_texts)
        ]
        # the answer is 5 <eos> <eos> <eos> <eos>
        # 0   0      0  0  1     1     1     1
        # 0   0      0  0  1     2     3     4
        # 0   0      0  0  0     1     2      3
        # T   T      T  T  T     F     F      F
        # True at EOS-token positions and False everywhere else
        # This means False at the generated output tokens and True at the first eos token and any other eos token after
        eos_tokens = generated_outputs.eq(tokenizer.eos_token_id)

        # both cumsum and eos_tokens.long will be zero until the first eos token, making the difference zero
        # and the mask True, 
        # at the first eos token, cumsum will be 1 and eos_tokens.long will be 1, which makes the difference still zero 
        # and the mask is True
        # beyond the first eos token, cumsum grows beyond 1 and eos_tokens.long remains 1 if the pad token is also the 
        # eos token, and when its not, eos_tokens.long is zero, therefore the difference
        # will not be zero, making that position False
        # Therefore, all positions including the first eos token is True and everything else is False
        first_eos_token = (eos_tokens.cumsum(dim=-1) - eos_tokens.long()) == 0

        generation_padding_mask = first_eos_token.to(outputs.dtype)

        label_masks = torch.zeros_like(outputs)
        label_masks[:, input_ids.shape[1]:] = generation_padding_mask

        input_attention_mask = torch.ones_like(outputs)

        # 0 on pad positions and 1 otherwise
        input_attention_mask[:, :input_ids.shape[1]] = attention_mask
        input_attention_mask[:, input_ids.shape[1]:] = generation_padding_mask

        batch_mean_reward = sum(rewards) / len(rewards)

        print(f"batch mean reward: {batch_mean_reward}")

        batch_metrics = {
            "batch": idx,
            "reward/train_batch_mean_reward": batch_mean_reward,
            # Completion length in unpadded tokens, to catch length collapse or
            # runaway generations early.
            "train/mean_completion_tokens": generation_padding_mask.sum(dim=-1).float().mean().item()
        }

        advantages = []

        for i in range(0, len(rewards), group_size):

            group_rewards = rewards[i: i + group_size]

            group_mean = sum(group_rewards) / len(group_rewards)

            group_advantages = [reward - group_mean for reward in group_rewards]

            advantages.extend(group_advantages)

        advantages = torch.tensor(advantages, device=outputs.device)[:, None]

        # Within-group spread of the reward. At zero every completion in a group
        # scored the same, the advantages vanish and the batch teaches nothing.
        batch_metrics["reward/advantage_abs_mean"] = advantages.abs().mean().item()

        if torch.all(advantages == 0):
            if trial + 1 > max_trials:
                print(f"Skipping batch {idx} due to zero advantages")
                batch_metrics["train/skipped_batch"] = 1
                batch_metrics["train/batch_trials"] = trial 
                wandb.log(batch_metrics)

                break         
            continue

        batch_metrics["train/skipped_batch"] = 0
        batch_metrics["train/batch_trials"] = trial 
        wandb.log(batch_metrics)

        old_token_log_probs = None

        input_ids = outputs[:, :-1]
        input_attention_mask = input_attention_mask[:, :-1]
        label_ids = outputs[:, 1:]
        label_masks = label_masks[:, 1:]

        for e in range(ppo_epoch):

            logits = model(input_ids=input_ids, attention_mask=input_attention_mask, use_cache=False,).logits

            new_log_probs = torch.log_softmax(logits, dim=-1)

            new_token_log_probs = torch.gather(
                new_log_probs,
                dim=-1,
                index=label_ids.unsqueeze(-1)
            ).squeeze(-1)

            if old_token_log_probs is None:
                old_token_log_probs = new_token_log_probs.detach()

            ratio = torch.exp(new_token_log_probs - old_token_log_probs)

            unclipped_loss = ratio * advantages
            clipped_loss = torch.clamp(ratio, min=1-eps_low, max=1+eps_high) * advantages

            ppo_objective = torch.min(unclipped_loss, clipped_loss)

            per_token_loss = -(ppo_objective * label_masks)

            sequence_loss = per_token_loss.sum(dim=-1) / max_new_tokens

            loss = sequence_loss.mean()

            optimizer.zero_grad()

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            
            wandb.log({
                "loss/step_loss": loss.item(),
                "grad/grad_norm": grad_norm.item(),
                "ppo_step": global_step
            })

            global_step += 1

            print(f"Step: {idx} PPO epoch: {e}, Loss: {loss.item()}")

        break


    if (idx + 1) % eval_steps == 0:
        evaluate(step=idx + 1)
        maths_evaluate(step=idx + 1)

  0%|          | 0/167 [00:00<?, ?it/s]

batch mean reward: 0.08333333333333333


  1%|          | 1/167 [00:34<1:35:03, 34.36s/it]

Step: 0 PPO epoch: 0, Loss: 0.0085177943110466
batch mean reward: 0.08333333333333333


  1%|          | 2/167 [00:41<50:45, 18.46s/it]  

...

Step: 106 PPO epoch: 0, Loss: -0.0021565761417150497
batch mean reward: 0.7291666666666666


 65%|██████▍   | 108/167 [17:49<06:36,  6.72s/it]

Step: 107 PPO epoch: 0, Loss: 0.005303274840116501
batch mean reward: 0.875


 65%|██████▌   | 109/167 [17:56<06:28,  6.69s/it]

Step: 108 PPO epoch: 0, Loss: 0.0049370634369552135


In [ ]:
# Close the run so wandb flushes everything and marks it finished
wandb.finish()